# CatBoost LTV-предсказание (v3)

Исправления vs v2:
- **GPU в Optuna** (было CPU → 308 минут!)
- **Feature Selection**: топ-150 фичей по importance из v2
- **stride=30** → честные фолды без перекрытия таргетов
- **Gini + RMSPE** метрики для tie-breaker
- **print callback** вместо зависающего виджета прогресса

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import optuna

FEATURES_DIR = Path("../data/processed/features_v3")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 4
N_OPTUNA_TRIALS = 15

def rmsle_score(y_true, y_pred):
    y_pred_clipped = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_clipped)))

def gini_normalized(y_true, y_pred):
    """Normalized Gini coefficient (tie-breaker метрика)."""
    def _gini(actual, predicted):
        n = len(actual)
        indices = np.argsort(-predicted)
        sorted_actual = actual[indices]
        cumulative = np.cumsum(sorted_actual)
        gini_sum = cumulative.sum() / sorted_actual.sum() - (n + 1) / 2
        return gini_sum / n
    return _gini(y_true, y_pred) / _gini(y_true, y_true)

def rmspe_total_gmv(y_true, y_pred):
    """RMSPE по суммарному GMV (tie-breaker)."""
    total_true = y_true.sum()
    total_pred = max(y_pred.sum(), 1e-8)
    return abs(total_true - total_pred) / total_true * 100

def load_fold(fold_path: Path) -> pd.DataFrame:
    df = pl.read_parquet(fold_path / "batch_*.parquet")
    return df.to_pandas()

## Feature Selection (топ-150 по importance из v3)

In [ ]:
TOP_N_FEATURES = 150

imp_path = Path("../data/processed/feature_importance_v3.json")
if imp_path.exists():
    with open(imp_path) as f:
        imp_data = json.load(f)
    ranked_features = [item["name"] for item in imp_data["features_ranked"]]
    print(f"Загружено ранжирование из v3: {len(ranked_features)} фичей")
else:
    ranked_features = None
    print("Ранжирование v3 не найдено, будут использованы все фичи")

print("Загрузка тестового фолда...")
test_df = load_fold(FEATURES_DIR / "fold_test")

drop_cols = ["user_id", "anchor_date", "target"]
all_features = [c for c in test_df.columns if c not in drop_cols]

if ranked_features:
    features = [f for f in ranked_features if f in all_features][:TOP_N_FEATURES]
else:
    features = all_features

print(f"Всего фичей в v3: {len(all_features)}")
print(f"Отобрано фичей: {len(features)}")
print(f"Топ-10: {features[:10]}")

test_predictions = np.zeros(len(test_df))
trained_models = []

## Optuna тюнинг (GPU + print callback + pruning)

In [ ]:
print("Загрузка фолдов...")
all_folds = []
for fold_idx in range(N_FOLDS):
    fold_df = load_fold(FEATURES_DIR / f"fold_{fold_idx:02d}")
    all_folds.append(fold_df)
    print(f"  Фолд {fold_idx}: {len(fold_df):,} записей")

print(f"\nЗапуск Optuna ({N_OPTUNA_TRIALS} триалов, GPU, усреднение по {N_FOLDS-1} фолдам)...\n")

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 1500, 4000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 5, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-2, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 42,
        'od_type': 'Iter',
        'early_stopping_rounds': 50,
        'verbose': False
    }
    
    scores = []
    for val_idx in range(1, N_FOLDS):
        train_dfs = [all_folds[i] for i in range(val_idx)]
        train_df = pd.concat(train_dfs, ignore_index=True)
        val_df = all_folds[val_idx]
        
        X_train = train_df[features]
        y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
        X_val = val_df[features]
        y_val = val_df["target"].values
        y_val_log = np.log1p(np.clip(y_val, 0, None))
        
        model = CatBoostRegressor(**params)
        model.fit(Pool(X_train, y_train_log), eval_set=Pool(X_val, y_val_log))
        
        val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
        scores.append(rmsle_score(y_val, val_pred))
        
        trial.report(np.mean(scores), val_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(scores)

def print_callback(study, trial):
    status = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"RMSLE={trial.value:.5f}"
    print(f"  Trial {trial.number+1:2d}/{N_OPTUNA_TRIALS}: {status} "
          f"(лучший: {study.best_value:.5f}, trial #{study.best_trial.number+1})")

study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[print_callback])

print(f"\n--- Итоги Optuna ---")
print(f"Лучший средний RMSLE: {study.best_value:.5f}")
print("Оптимальные параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Cumulative Walk-Forward с оптимальными параметрами

In [ ]:
best_params = study.best_params

print("Финальное обучение с Optuna параметрами...\n")

fold_scores = []
fold_gini = []
fold_rmspe = []

for val_idx in range(1, N_FOLDS):
    train_dfs = [all_folds[i] for i in range(val_idx)]
    train_df = pd.concat(train_dfs, ignore_index=True)
    val_df = all_folds[val_idx]
    
    train_str = "+".join([str(i) for i in range(val_idx)])
    print(f"--- Train: [{train_str}] ({len(train_df):,}) | Val: fold {val_idx} ({len(val_df):,}) ---")
    
    X_train = train_df[features]
    y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
    X_val = val_df[features]
    y_val = val_df["target"].values
    y_val_log = np.log1p(np.clip(y_val, 0, None))

    model = CatBoostRegressor(
        **best_params,
        loss_function='RMSE',
        eval_metric='RMSE',
        task_type='GPU',
        devices='0',
        random_seed=42,
        early_stopping_rounds=150,
        verbose=250
    )
    
    model.fit(Pool(X_train, y_train_log), eval_set=Pool(X_val, y_val_log))
    trained_models.append(model)
    
    val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
    
    score = rmsle_score(y_val, val_pred)
    gini = gini_normalized(y_val, val_pred)
    rmspe = rmspe_total_gmv(y_val, val_pred)
    
    fold_scores.append(score)
    fold_gini.append(gini)
    fold_rmspe.append(rmspe)
    
    print(f"RMSLE: {score:.5f} | Gini: {gini:.4f} | RMSPE total GMV: {rmspe:.2f}%\n")
    
    test_pred_log = model.predict(test_df[features])
    test_predictions += np.expm1(np.clip(test_pred_log, 0, None)) / (N_FOLDS - 1)

    model.save_model(MODELS_DIR / f"catboost_v3_fold_{val_idx}.cbm")

print("="*60)
print(f"Средний RMSLE: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")
print(f"Средний Gini:  {np.mean(fold_gini):.4f}")
print(f"Средний RMSPE: {np.mean(fold_rmspe):.2f}%")

## Feature Importance и Сабмит

In [ ]:
importances = trained_models[-1].get_feature_importance()
imp_dict = dict(zip(features, importances))
sorted_imp = sorted(imp_dict.items(), key=lambda x: x[1], reverse=True)

print("Топ-25 признаков:")
for feat, imp in sorted_imp[:25]:
    print(f"  {feat}: {imp:.2f}%")

fig, ax = plt.subplots(figsize=(10, 8))
top_n = 30
top_feats = sorted_imp[:top_n]
ax.barh([f[0] for f in reversed(top_feats)], [f[1] for f in reversed(top_feats)], color='#2196F3')
ax.set_xlabel('Важность (%)')
ax.set_title(f'Топ-{top_n} признаков CatBoost v3')
plt.tight_layout()
plt.show()

submit_df = test_df[["user_id"]].copy()
submit_df["predict"] = np.clip(test_predictions, 0, None)

submit_path = Path("../data/processed/catboost_v3_submission.csv")
submit_df.to_csv(submit_path, index=False)

print(f"\nСохранено: {submit_path}")
print(f"mean={submit_df['predict'].mean():.2f} | median={submit_df['predict'].median():.2f} | max={submit_df['predict'].max():.2f}")

v3_imp = {"features_ranked": [{"name": f, "importance": round(imp, 4)} for f, imp in sorted_imp]}
with open("../data/processed/feature_importance_v3.json", "w") as f:
    json.dump(v3_imp, f, indent=2)
print("Feature importance v3 сохранено.")